In [1]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI')

In [2]:
!pip install langchain-community pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.3/554.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 1.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.0
    Uninstalling langchain-core-1.4.0:
      Successfully uninstalled langchain-core-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is inco

In [3]:
!pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 801.9 kB/s eta 0:00:00


In [4]:
!pip install neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 1.0 MB/s eta 0:00:00


In [ ]:
## Graph
Entities --> Nodes
Relationships --> Edges

In [ ]:
Elon Musk --CEO of--> Tesla --revenue is --> 5B$
 |
CEO of
 |
SpaceX

In [ ]:
# Neo4J --> DB to store graph data

##GraphRAG :

In [ ]:
# Neo4J AuraDB --> Managed service of neo4j

>> Cypher Query --> To interact with neo4j

In [ ]:
# https://console.neo4j.io/  --> sign in
>> Create instance :
>> Download credentials

In [ ]:
## Downloaded Credentials conetnst :
"""
# Wait 60 seconds before connecting using these details, or login to https://console.neo4j.io to validate the Aura Instance is available
NEO4J_URI=neo4j+s://7c7c6bb6.databases.neo4j.io
NEO4J_USERNAME=7c7c6bb6
NEO4J_PASSWORD=zUTv8ZjBflzOOmOHQITct5WCH_QKywUNgiLYanqqIGw
NEO4J_DATABASE=7c7c6bb6
AURA_INSTANCEID=7c7c6bb6
AURA_INSTANCENAME=Instance01

"""

In [5]:
# import os
# os.environ["NEO4J_URI"] = "neo4j+s://<your-db>.databases.neo4j.io"
# os.environ["NEO4J_USERNAME"] = "neo4j"
# os.environ["NEO4J_PASSWORD"] = "password"

NEO4J_URI = "neo4j+s://7c7c6bb6.databases.neo4j.io"
NEO4J_USERNAME = "7c7c6bb6"
NEO4J_PASSWORD = "zUTv8ZjBflzOOmOHQITct5WCH_QKywUNgiLYanqqIGw"
NEO4J_DATABASE = "7c7c6bb6"

In [6]:
# Load Data (Same as Normal RAG)
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/tsla (1).pdf")
pages = loader.load()

/tmp/ipykernel_5954/1076238814.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [7]:
# Split Text (Same as Normal RAG)
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

documents = splitter.split_documents(pages)

In [8]:
# Initialize LLM
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0
)


In [9]:
!pip install langchain-experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 5.4 MB/s eta 0:00:00


In [10]:
# MAGIC STEP — Text → Knowledge Graph (NO Cypher)
from langchain_experimental.graph_transformers import LLMGraphTransformer

graph_transformer = LLMGraphTransformer(llm=llm) # to get entity and relations

graph_documents = graph_transformer.convert_to_graph_documents(documents)

/tmp/ipykernel_5954/4289858925.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


In [12]:
graph_documents[0].nodes

[Node(id='United States Securities And Exchange Commission', type='Organization', properties={}),
 Node(id='Form 10-K', type='Document', properties={}),
 Node(id='December 31, 2023', type='Date', properties={}),
 Node(id='001-34756', type='Identifier', properties={}),
 Node(id='Tesla, Inc.', type='Organization', properties={}),
 Node(id='Delaware', type='Location', properties={}),
 Node(id='91-2197729', type='Identifier', properties={}),
 Node(id='1 Tesla Road, Austin, Texas, 78725', type='Location', properties={}),
 Node(id='(512) 516-8177', type='Contact', properties={})]

In [14]:
graph_documents[0].relationships

[Relationship(source=Node(id='Form 10-K', type='Document', properties={}), target=Node(id='United States Securities And Exchange Commission', type='Organization', properties={}), type='ISSUED_BY', properties={}),
 Relationship(source=Node(id='Form 10-K', type='Document', properties={}), target=Node(id='December 31, 2023', type='Date', properties={}), type='FOR_FISCAL_YEAR_ENDED', properties={}),
 Relationship(source=Node(id='Form 10-K', type='Document', properties={}), target=Node(id='001-34756', type='Identifier', properties={}), type='COMMISSION_FILE_NUMBER', properties={}),
 Relationship(source=Node(id='Tesla, Inc.', type='Organization', properties={}), target=Node(id='Delaware', type='Location', properties={}), type='INCORPORATED_IN', properties={}),
 Relationship(source=Node(id='Tesla, Inc.', type='Organization', properties={}), target=Node(id='91-2197729', type='Identifier', properties={}), type='EMPLOYER_IDENTIFICATION_NUMBER', properties={}),
 Relationship(source=Node(id='Tesla

In [11]:
graph_documents[0]

GraphDocument(nodes=[Node(id='United States Securities And Exchange Commission', type='Organization', properties={}), Node(id='Form 10-K', type='Document', properties={}), Node(id='December 31, 2023', type='Date', properties={}), Node(id='001-34756', type='Identifier', properties={}), Node(id='Tesla, Inc.', type='Organization', properties={}), Node(id='Delaware', type='Location', properties={}), Node(id='91-2197729', type='Identifier', properties={}), Node(id='1 Tesla Road, Austin, Texas, 78725', type='Location', properties={}), Node(id='(512) 516-8177', type='Contact', properties={})], relationships=[Relationship(source=Node(id='Form 10-K', type='Document', properties={}), target=Node(id='United States Securities And Exchange Commission', type='Organization', properties={}), type='ISSUED_BY', properties={}), Relationship(source=Node(id='Form 10-K', type='Document', properties={}), target=Node(id='December 31, 2023', type='Date', properties={}), type='FOR_FISCAL_YEAR_ENDED', properties

In [ ]:
# `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``

In [15]:
from langchain_community.graphs import Neo4jGraph

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)

graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)

/tmp/ipykernel_5954/2013615990.py:3: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(


In [16]:
!pip install langchain-neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.3/262.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 2.3 MB/s eta 0:00:00


In [ ]:
## Normal RAG :
Query --> VectorDB --> similarity --> relevent docs

## GraphRAG :
Query --> VectorDB --> similarity --> relevent docs
Query --> Extract entity --> GraphDB -->  relevent docs

In [ ]:
"Who is CEO of Tesla?"  --> CEO, Tesla --> fetch relevent context from these two nodes --> LLM

In [17]:
# Create Vector Store ON TOP of the Graph
from langchain_neo4j import Neo4jVector
from langchain_openai import OpenAIEmbeddings

vector_store = Neo4jVector.from_existing_graph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,

    embedding=OpenAIEmbeddings(),
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
    search_type="hybrid"
    )


In [18]:
# Entity-Aware Retriever (LangChain Only)
from langchain_core.documents import Document

def graph_rag_retriever(question: str) -> str:
    docs = vector_store.similarity_search(question, k=4)
    return "\n\n".join([d.page_content for d in docs])


In [19]:
# Final GraphRAG Prompt
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the question using ONLY the following context.
Context:
{context}

Question:
{question}

Answer in simple language.
""")


In [20]:
# Build GraphRAG Chain (Looks Like Normal RAG!)
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

graph_rag_chain = (
    {
        "context": graph_rag_retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [21]:
graph_rag_chain.invoke("Who is CEO of Tesla?")

'The context provided does not mention the CEO of Tesla.'

In [22]:
graph_rag_chain.invoke("What is revenue of tesla?")

"Tesla's total revenue is $96,773 million."

In [23]:
graph_rag_chain.invoke("What is telephone of tesla?")

'The telephone number of Tesla is (512) 516-8177.'

In [24]:
graph_rag_chain.invoke("What is address of tesla?")

'The address of Tesla is 1 Tesla Road, Austin, Texas 78725.'